In [1]:
## conda env!!!

In [2]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method
METHOD = 'mhcnuggets'
ORGANISMS_TO_USE = ['Human']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_A_02_01',
 'HLA_B_27_05',
 'HLA_A_03_01',
 'HLA_B_07_02',
 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05']

In [3]:
# importing the predict module
from mhcnuggets.src.predict import predict
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import numpy as np
import pandas as pd

Using TensorFlow backend.


In [4]:
# Read datasets
DECOY_KEY = 'x1'

DATA_PATH = "/home/denklewer/Data/"
INPUT_DATA_PATH = DATA_PATH + 'input/'
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'
per_allele_val_df = {
    allele_name :pd.read_csv(f"{INPUT_DATA_PATH}df_{allele_name}_{DECOY_KEY}_val_for_classification.csv", index_col=0) for allele_name in ALLELES
}



In [5]:
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)

assert not 0 in [len(per_allele_val_df[allele_name]) for allele_name in ALLELES], 'One of dfs has length of 0, check data reading'


/home/denklewer/Tools/anacoda3/envs/mhcnuggets/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3361: DtypeWarning: Columns (6,7,8,9,14,21,22,23,24) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


In [6]:
df = per_allele_val_df[ALLELES[0]]
df.head()

,split,el,allele,peptide,n_flank,c_flank,mhc_a_1,mhc_a_2,mhc_b_1,mhc_b_2,...,gene_id,gene_symbol,n_samples,n_studies,Sample_ids,sample_names,Study_ids,study_names,negative_ensembl_id,binder
52052,test,1,HLA-A*02:01,AGIALNDNFVK,DTHSSIFDAK,LISWYDNEYG,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000105679,GAPDHS,1.0,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1
52053,test,1,HLA-A*02:01,AGLYRGLCAVRSR,GGAGPAVLQV,ALGLGLVSPA,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000123472,ATPAF1,1.0,1.0,1320,iedb_1029626_EL_A0201,5,iedb_consolidated,NaN,1
52054,test,1,HLA-A*02:01,AIDEKVQAV,APDIDKLKMA,HNLKMFWQST,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000100154,TTC28,1.0,1.0,1887,iedb_1034377_EL_A0201,5,iedb_consolidated,NaN,1
52055,test,1,HLA-A*02:01,AIFSAIIFL,VRHCFFTVIF,SFEAVSSLSK,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000005486,RHBDD2,1.0,1.0,677,Sarkizova_A0201,21,sarkizova_mhcI_2020,NaN,1
52056,test,1,HLA-A*02:01,AIIGGTFTV,PLYRFITTIC,AGILDSCIFT,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,YFAMYGEKVAHTHVDTLYVRYHYYTWAVLAYTWY,NaN,NaN,...,ENSG00000113719,ERGIC1,2.0,1.0,"1356,1887","iedb_1019570_EL_A0201,iedb_1034377_EL_A0201",5,iedb_consolidated,NaN,1


In [7]:
def defineClass(allele_name):
    for organism_to_use in ORGANISMS_TO_USE:
        for class_to_use in CLASSES_TO_USE:
            if allele_name in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]:
                return class_to_use

In [8]:
# add ID, add length, filter by length, rename label column
for allele_name in ALLELES:
    per_allele_val_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_val_df[allele_name].index.astype(str).values
    per_allele_val_df[allele_name]['length'] = per_allele_val_df[allele_name]['peptide'].str.len()
    #Filter for mhcnuggets <=15 length for class 1 and <=30 for class 2
    initial_len = len(per_allele_val_df[allele_name])
    if defineClass(allele_name) == 'I':
        
        per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 15]
        
    else:
        per_allele_val_df[allele_name] = per_allele_val_df[allele_name][per_allele_val_df[allele_name]['length'] <= 30]
    result_len = len(per_allele_val_df[allele_name])
    print(f"Filtered out {initial_len - result_len} samples")


Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples
Filtered out 0 samples


## Prediction code

In [9]:
# different allele naming.
TARGET_ALLELES = dict()

## HARDCODE
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    elif allele_name == 'HLA_DRB1_01_01':
        target_allele_name = 'HLA-DRB101:01'
    elif allele_name == 'HLA_DRB1_04_01':
        target_allele_name = 'HLA-DRB104:01'
    elif allele_name == 'HLA_DRB1_04_05':
        target_allele_name = 'HLA-DRB104:05'    
    else:
        target_allele_name = normalize_allele_name(allele_name).replace("*", "")
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA_A_02_01': 'HLA-A02:01',
 'HLA_B_27_05': 'HLA-B27:05',
 'HLA_A_03_01': 'HLA-A03:01',
 'HLA_B_07_02': 'HLA-B07:02',
 'HLA_DRB1_01_01': 'HLA-DRB101:01',
 'HLA_DRB1_04_01': 'HLA-DRB104:01',
 'HLA_DRB1_04_05': 'HLA-DRB104:05'}

In [16]:
# predicting new line separated peptides present in the peptides_path file 
# for MHC class_I allele HLA-A02:01. Since a MS_BA blend model is available,
# this prediction will use the blended model (HLA-A02:01_BA_to_HLAp.h5)
def predict_with_mhcnuggets(per_allele_df):
    # save to temp dir
    TEMPDIR = 'TMP/'
    for allele_name in ALLELES:
        per_allele_val_df[allele_name]['peptide'].to_csv(f"{TEMPDIR}mchnuggets_data_{allele_name}", index=False, header=False)    
    # predict
    for allele_name in ALLELES:
        class_to_use = defineClass(allele_name)
        target_allele = TARGET_ALLELES[allele_name]
        print(f"----------{target_allele}")
        predict(class_=f'{class_to_use}',
                peptides_path=f'{TEMPDIR}mchnuggets_data_{allele_name}', 
                mhc=target_allele,
                model_weights_path=f"Models/mhcnuggets/{target_allele}_BA.h5",
                ba_models=True,
                output=f'{TEMPDIR}mchnuggets_predictions_{allele_name}')
    # read predictions from temp
    per_allele_predictions= {
        allele_name :pd.read_csv(f"{TEMPDIR}mchnuggets_predictions_{allele_name}") for allele_name in ALLELES
    }
    for allele_name in per_allele_predictions.keys(): 
        df_result = per_allele_predictions[allele_name]
        # add binder label
        
        per_allele_predictions[allele_name] = df_result.merge(per_allele_df[allele_name][[RESULT_COLUMNS['id'],RESULT_COLUMNS['label'], RESULT_COLUMNS['peptide']]], on=['peptide'])
        # log-scale affinity
        
        per_allele_predictions[allele_name]['score'] = 1 - np.log(per_allele_predictions[allele_name]['ic50'])/np.log(50000)
    return per_allele_predictions
    

## create tempfiles for mhcnuggets



In [17]:
per_allele_predictions = predict_with_mhcnuggets(per_allele_val_df)

----------HLA-A02:01
Predicting for 68900 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-A02:01
Predicting with user-specified model: Models/mhcnuggets/HLA-A02:01_BA.h5
----------HLA-B27:05
Predicting for 169100 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B27:05
Predicting with user-specified model: Models/mhcnuggets/HLA-B27:05_BA.h5
----------HLA-A03:01
Predicting for 18300 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-A03:01
Predicting with user-specified model: Models/mhcnuggets/HLA-A03:01_BA.h5
----------HLA-B07:02
Predicting for 33000 peptides
Number of peptides skipped/total due to length 0 / 0
Building model
Closest allele found HLA-B07:02
Predicting with user-specified model: Models/mhcnuggets/HLA-B07:02_BA.h5
----------HLA-DRB101:01
Predicting for 23659 peptides
Number of peptides skipped/total due to length 0 

In [18]:
assert RESULT_COLUMNS['label'] in per_allele_predictions[ALLELES[0]].columns, f"label column must be in the result, this column is '{RESULT_COLUMNS['label']}'"
assert RESULT_COLUMNS['id'] in per_allele_predictions[ALLELES[0]].columns, f"id column must be in the result, this column is '{RESULT_COLUMNS['id']}'"
assert RESULT_COLUMNS['peptide'] in per_allele_predictions[ALLELES[0]].columns, f"peptide column must be in the result, this column is '{RESULT_COLUMNS['peptide']}'"

#### write to files

In [19]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_predictions.items():
    total_df.to_csv(f"{TARGET_PATH_TO_PREDICTIONS}{METHOD}_predictions_{allele_name}")

In [20]:
for allele_name in ALLELES:
    df = per_allele_predictions[allele_name]
    assert not df.isnull().values.any()
    assert not df.isna().values.any()
    assert np.isfinite(df['score']).all()
#     print(np.isnan().value_counts())
#     print(np.isnan(df['score']).value_counts())
#     print(np.isinf(df['score']).value_counts())